# float32 is OPT-IN

The switch below is `False`: this notebook runs in float64, the default. float32 is **off by default and is not a safe replacement for float64**.

- **Set it before import.** The switch below sets PyTensor's `floatX` flag, and PyTensor reads it when it is imported. If you change `USE_FLOAT32`, **restart the kernel** and run the notebook from the top. Setting it after `import pymc` does nothing.
- **Where it helps.** On a GPU, models with a lot of arithmetic per step, such as logistic regression and the non-centered radon model, run 3x to 8x faster (see `float32.ipynb`). On a CPU expect little.
- **Where it fails.** The GEV model: the step size collapses and the chains barely move. 8 Schools failed the bias gate on 2 of 13 seeds. The failure can be silent.
- **Three checks on every float32 result** (README, section "float32 on the GPU: opt-in, and check the result"):
  1. `idata.attrs["frozen_step_size"]`: orders of magnitude below the float64 value is the GEV failure;
  2. R-hat on every parameter: far above 1.01 is a failure. (With only 10 draws per chain R-hat is inflated even in float64, so compare with a float64 run of the same model; a collapsed sampler is orders of magnitude worse, not tenths.)
  3. `sample_stats["num_integration_steps"]`: hundreds of steps instead of tens.

Do **not** enable `jax_enable_x64` yourself; PyTensor manages it from `floatX`. GEV is the model that **fails** under float32; keep this notebook in float64.

In [ ]:
USE_FLOAT32 = False      # the switch: read the cell above, then restart the kernel to change it

import os
if USE_FLOAT32:
    os.environ["PYTENSOR_FLAGS"] = "floatX=float32,blas__ldflags=-llapack -lblas -lcblas -lm"
print("float32 requested:", USE_FLOAT32)

# Extreme value example
Based on the Port Pirie example in Coles (2001). See also https://www.pymc.io/projects/examples/en/latest/case_studies/GEV.html

Requires `pymc-extras`: https://www.pymc.io/projects/extras/en/latest/

In [ ]:
%matplotlib inline
CHAINS, DRAWS, WARMUP = 1024, 10, 25      # many short chains; WARMUP is chees_kwargs["draws"]
SEED = 42

import jax
import numpy as np
import pymc as pm
import arviz as az
import pytensor
from pfcheeshmc import sample_pf_chees_hmc

print("JAX devices:", jax.devices(), "| floatX:", pytensor.config.floatX, "| x64:", jax.config.jax_enable_x64)
import matplotlib.pyplot as plt
import pymc_extras as pmx
plt.rcParams["figure.dpi"] = 72      # keep the stored plots small

In [ ]:
# fmt: off
data = np.array([4.03, 3.83, 3.65, 3.88, 4.01, 4.08, 4.18, 3.80,
                 4.36, 3.96, 3.98, 4.69, 3.85, 3.96, 3.85, 3.93,
                 3.75, 3.63, 3.57, 4.25, 3.97, 4.05, 4.24, 4.22,
                 3.73, 4.37, 4.06, 3.71, 3.96, 4.06, 4.55, 3.79,
                 3.89, 4.11, 3.85, 3.86, 3.86, 4.21, 4.01, 4.11,
                 4.24, 3.96, 4.21, 3.74, 3.85, 3.88, 3.66, 4.11,
                 3.71, 4.18, 3.90, 3.78, 3.91, 3.72, 4.00, 3.66,
                 3.62, 4.33, 4.55, 3.75, 4.08, 3.90, 3.88, 3.94,
                 4.33])
# fmt: on
plt.figure(figsize=(5, 3)); plt.hist(data); plt.xlabel("annual maximum sea level (m)"); plt.show()

In [ ]:
p = 1 / 10  # return period
with pm.Model() as model:
    # Priors
    μ = pm.Normal("μ", mu=3.8, sigma=0.2)
    σ = pm.HalfNormal("σ", sigma=0.3)
    ξ = pm.TruncatedNormal("ξ", mu=0, sigma=0.2, lower=-0.6, upper=0.6)

    # Likelihood
    gev = pmx.GenExtreme("gev", mu=μ, sigma=σ, xi=ξ, observed=data)
    # Return level
    z_p = pm.Deterministic("z_p", μ - σ / ξ * (1 - (-np.log(1 - p)) ** (-ξ)))

In [ ]:
idata = pm.sample_prior_predictive(samples=500, model=model, random_seed=SEED)
az.plot_ppc(idata, group="prior", figsize=(8, 4))
ax = plt.gca(); ax.set_xlim([2, 6]); ax.set_ylim([0, 2]);

In [ ]:
with model:
    posterior_idata = sample_pf_chees_hmc(draws=DRAWS, chains=CHAINS, random_seed=SEED, progressbar=False,
                                          chees_kwargs={"draws": WARMUP})
print("frozen step size:", posterior_idata.attrs["frozen_step_size"])
idata.extend(posterior_idata)
az.summary(posterior_idata)

In [ ]:
# Plot 8 of the 1,024 chains.
az.plot_trace(posterior_idata.posterior.isel(chain=slice(0, 8)), compact=True);

Posterior predictive from 100 of the chains (1,000 draws).

In [ ]:
with model:
    post_idata = pm.sample_posterior_predictive(posterior_idata.posterior.isel(chain=slice(0, 100)),
                                                random_seed=SEED, progressbar=False)
idata.extend(post_idata)
az.plot_dist_comparison(idata, var_names=["μ", "σ", "ξ"]);

# NUTS
Compare with PyMC's NUTS (4 chains, 1,000 draws each): the posterior summaries should agree.

In [ ]:
with model:
    nuts_idata = pm.sample(draws=1000, chains=4, tune=1000, target_accept=0.95, random_seed=SEED, progressbar=False)
import pandas as pd
cols = ["mean", "sd", "hdi_3%", "hdi_97%"]
pd.concat({"pf-chees-hmc": az.summary(posterior_idata, var_names=["μ", "σ", "ξ", "z_p"])[cols],
           "NUTS": az.summary(nuts_idata, var_names=["μ", "σ", "ξ", "z_p"])[cols]}, axis=1)

In [ ]:
az.plot_trace(nuts_idata, var_names=["μ", "σ", "ξ"]);